In [ ]:
"""
Module 1: Traditional Similarity Metrics

Implements evaluation metrics that compare AI responses against golden reference
responses using cosine similarity, BLEU scores, and ROUGE-L scores.
"""

from typing import List, Dict, Any
import numpy as np
from sentence_transformers import SentenceTransformer


def calculate_cosine_similarity(text1: str, text2: str, model: SentenceTransformer) -> float:
    """
    Calculate cosine similarity between two texts using sentence embeddings.

    Converts both texts to embeddings using the provided sentence transformer model,
    then computes the cosine similarity between the embedding vectors.

    Args:
        text1: First text to compare
        text2: Second text to compare
        model: SentenceTransformer model for generating embeddings

    Returns:
        Cosine similarity score between 0 and 1 (1 = identical meaning)

    Edge Cases:
        - Empty strings should return 0.0
        - None values should return 0.0
    """
    if not text1 or not text2:
        return 0.0

    embedding1 = model.encode(text1, convert_to_tensor=True)
    embedding2 = model.encode(text2, convert_to_tensor=True)

    cosine_sim = np.dot(embedding1, embedding2) / (np.linalg.norm(embedding1) * np.linalg.norm(embedding2))
    return float(cosine_sim)


def calculate_bleu_score(reference: str, candidate: str, max_n: int = 4) -> float:
    """
    Calculate BLEU score measuring n-gram overlap between texts.

    Implements the BLEU metric which calculates precision for n-grams (1 to max_n)
    and combines them with a brevity penalty. BLEU is commonly used for evaluating
    machine translation and text generation quality.

    Formula:
    - For each n in 1 to max_n: calculate precision of n-grams
    - Geometric mean of all precisions
    - Apply brevity penalty: BP = exp(1 - ref_len/cand_len) if cand_len < ref_len else 1.0
    - BLEU = BP * geometric_mean

    Args:
        reference: Reference (golden) text
        candidate: Generated text to evaluate
        max_n: Maximum n-gram size (default 4)

    Returns:
        BLEU score between 0 and 1 (1 = perfect match)

    Edge Cases:
        - Empty strings should return 0.0
        - None values should return 0.0
        - If any n-gram precision is 0, return 0.0
    """
    if not reference or not candidate:
        return 0.0

    reference_tokens = reference.split()
    candidate_tokens = candidate.split()

    precisions = []
    for n in range(1, max_n):
        reference_ngrams = set(tuple(reference_tokens[i:i+n]) for i in range(len(reference_tokens)-n+1))
        candidate_ngrams = set(tuple(candidate_tokens[i:i+n]) for i in range(len(candidate_tokens)-n+1))
        if not candidate_ngrams:
            precisions.append(0.0)
            continue
        precision = len(reference_ngrams & candidate_ngrams) / len(candidate_ngrams)
        precisions.append(precision)

    if not precisions:
        return 0.0

    geometric_mean = np.exp(np.mean(np.log(precisions))) if all(precisions) else 0.0
    cand_len = len(candidate_tokens)
    ref_len = len(reference_tokens)
    brevity_penalty = np.exp(1 - ref_len/cand_len) if cand_len < ref_len else 1.0
    bleu_score = brevity_penalty * geometric_mean
    return float(bleu_score)


def calculate_rouge_l(reference: str, candidate: str) -> float:
    """
    Calculate ROUGE-L score using longest common subsequence.

    ROUGE-L measures the longest common subsequence (LCS) between reference and
    candidate text, providing a recall-oriented metric that captures content overlap
    without requiring consecutive matches.

    Formula:
    - Find LCS length between reference and candidate
    - Precision = LCS_length / candidate_length
    - Recall = LCS_length / reference_length
    - F1 = 2 * (Precision * Recall) / (Precision + Recall)

    Args:
        reference: Reference (golden) text
        candidate: Generated text to evaluate

    Returns:
        ROUGE-L F1 score between 0 and 1 (1 = perfect match)

    Edge Cases:
        - Empty strings should return 0.0
        - None values should return 0.0
        - If precision + recall = 0, return 0.0
    """
    if not reference or not candidate:
        return 0.0

    def lcs_length(seq1: List[str], seq2: List[str]):
        if len(seq1) < len(seq2):
            seq1, seq2 = seq2, seq1

        m, n = len(seq1), len(seq2)
        dp = [0] * (n + 1)

        for i in range(1, m + 1):
            prev_diag = 0
            for j in range(1, n + 1):
                curr = dp[j]
                if seq1[i - 1] == seq2[j - 1]:
                    dp[j] = prev_diag + 1
                else:
                    dp[j] = max(dp[j], dp[j - 1])
                prev_diag = curr

        return dp[n]

    reference_tokens = reference.split()
    candidate_tokens = candidate.split()
    lcs_len = lcs_length(reference_tokens, candidate_tokens)
    precision = lcs_len / len(candidate_tokens) if candidate_tokens else 0.0
    recall = lcs_len / len(reference_tokens) if reference_tokens else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0.0
    return f1

def evaluate_with_multiple_metrics(
    reference: str,
    candidate: str,
    model: SentenceTransformer,
    weights: Dict[str, float] = None
) -> Dict[str, float]:
    """
    Evaluate response using multiple metrics with weighted combination.

    Combines cosine similarity, BLEU score, and ROUGE-L score to provide
    comprehensive evaluation. Returns individual metric scores and a weighted
    overall score.

    Default weights:
    - cosine_similarity: 0.4 (semantic meaning)
    - bleu_score: 0.3 (precision/word choice)
    - rouge_l: 0.3 (content recall)

    Args:
        reference: Reference (golden) text
        candidate: Generated text to evaluate
        model: SentenceTransformer model for embeddings
        weights: Optional custom weights for metrics (must sum to 1.0)

    Returns:
        Dictionary with:
        - 'cosine_similarity': float
        - 'bleu_score': float
        - 'rouge_l': float
        - 'overall_score': float (weighted combination, scaled 0-10)

    Edge Cases:
        - If weights provided, validate they sum to 1.0 (±0.01 tolerance)
        - If weights invalid, use default weights
        - Empty/None inputs should return all scores as 0.0
    """
    # Default weights
    default_weights = {
        'cosine_similarity': 0.4,
        'bleu_score': 0.3,
        'rouge_l': 0.3
    }

    # Validate and use provided weights if valid
    if weights:
        total_weight = sum(weights.values())
        if abs(total_weight - 1.0) > 0.01:
            print("Warning: Provided weights do not sum to 1. Using default weights.")
            weights = default_weights
    else:
        weights = default_weights

    # Calculate individual metrics
    cosine_sim = calculate_cosine_similarity(reference, candidate, model)
    bleu = calculate_bleu_score(reference, candidate)
    rouge = calculate_rouge_l(reference, candidate)

    # Calculate overall score using weighted combination
    overall_score = (
        weights['cosine_similarity'] * cosine_sim +
        weights['bleu_score'] * bleu +
        weights['rouge_l'] * rouge
    ) * 10  # Scale to 0-10

    return {
        'cosine_similarity': cosine_sim,
        'bleu_score': bleu,
        'rouge_l': rouge,
        'overall_score': overall_score
    }


In [ ]:
"""
Module 2: LLM-as-a-Judge Evaluation

Implements LLM-based evaluation system that provides nuanced quality assessment
with structured output, multi-dimensional scoring, and quality classification.
"""

from typing import Dict, Any, Optional
import json
import re


def create_judge_prompt(query: str, response: str) -> str:
    """
    Create a structured judge prompt with clear evaluation criteria and rubrics.

    The prompt should instruct an LLM to evaluate a customer service response across
    multiple dimensions: acccreate_judge_prompturacy, helpfulness, professionalism, and clarity.

    Prompt should include:
    - Clear role definition for the judge
    - The customer query and AI response to evaluate
    - Specific evaluation criteria with 1-10 scoring scales
    - Rubric examples showing what constitutes high vs low scores
    - Request for JSON output format

    Expected JSON format:
    {
        "accuracy": {"score": 1-10, "justification": "..."},
        "helpfulness": {"score": 1-10, "justification": "..."},
        "professionalism": {"score": 1-10, "justification": "..."},
        "clarity": {"score": 1-10, "justification": "..."},
        "overall_score": 1-10
    }

    Args:
        query: The customer's original question/request
        response: The AI-generated response to evaluate

    Returns:
        Formatted judge prompt string
    """

    judge_prompt = f"""
    You are an expert evaluator assessing AI-generated content.

    Original request: {query}
    AI response: {response}

    You will evaluate the AI response based on the following criteria:
    - Accuracy: How correct and relevant is the information?
    - Helpfulness: How useful is the response to the user?
    - Professionalism: Is the tone appropriate and respectful?
    - Clarity: Is the response clear and easy to understand?

    Please provide your evaluation in the following JSON format:
    {{
        "accuracy": {{"score": "1-10", "justification": "..."}},
        "helpfulness": {{"score": "1-10", "justification": "..."}},
        "professionalism": {{"score": "1-10", "justification": "..."}},
        "clarity": {{"score": "1-10", "justification": "..."}},
        "overall_score": "1-10"
    }}
    """

    return judge_prompt


def parse_judge_response(judge_output: str) -> Dict[str, Any]:
    """
    Parse structured output from LLM judge into dictionary format.

    Extracts JSON from the judge's response, handling cases where the judge
    includes additional text before/after the JSON structure. Uses regex to
    find JSON blocks and parses them.

    Expected structure:
    {
        "accuracy": {"score": int, "justification": str},
        "helpfulness": {"score": int, "justification": str},
        "professionalism": {"score": int, "justification": str},
        "clarity": {"score": int, "justification": str},
        "overall_score": int
    }

    Args:
        judge_output: Raw text output from LLM judge

    Returns:
        Parsed dictionary with evaluation data

    Edge Cases:
        - If no JSON found, return empty dict with error flag
        - If JSON is malformed, return empty dict with error flag
        - If required fields missing, return partial data with error flag
        - Validate score ranges (1-10), set to 5 if out of range
    """
    json_regex = r"\{.*\}"
    match = re.search(json_regex, judge_output, re.DOTALL)

    if not match:
        return {
            "error": "No valid JSON found",
            "message": "Failed to extract JSON from judge output",
        }

    json_str = match.group(0)
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError:
        return {
            "error": "Malformed JSON",
            "message": "Failed to parse JSON from judge output",
        }

    required_keys = [
        "accuracy",
        "helpfulness",
        "professionalism",
        "clarity",
        "overall_score",
    ]
    if not all(k in data for k in required_keys):
        data["error"] = "Missing required fields"

    for key in required_keys:
        if key in data:
            if isinstance(data[key], dict):
                score = data[key].get("score", 5)
                if not isinstance(score, (int, float)) or not (1 <= score <= 10):
                    data[key]["score"] = 5
            elif key == "overall_score":
                score = data[key]
                if not isinstance(score, (int, float)) or not (1 <= score <= 10):
                    data["overall_score"] = 5
        else:
            data[key] = {"score": 5, "justification": "Missing"}

    return data


def evaluate_response_quality(
    query: str, response: str, judge_model_func: Any
) -> Dict[str, Any]:
    """
    Run complete LLM judge evaluation on a response.

    Orchestrates the full evaluation process:
    1. Create judge prompt with criteria
    2. Call judge model to get evaluation
    3. Parse structured output
    4. Return formatted results

    Args:
        query: Customer's original question
        response: AI-generated response to evaluate
        judge_model_func: Callable that takes prompt and returns judge response
                         Should be: func(prompt: str) -> str

    Returns:
        Dictionary with:
        - 'dimensions': Dict of dimension scores and justifications
        - 'overall_score': Overall quality score (1-10)
        - 'evaluation_success': Boolean indicating if evaluation completed

    Edge Cases:
        - Handle judge model errors gracefully
        - If parsing fails, return error indication
        - Ensure all scores are in valid range (1-10)
    """
    prompt = create_judge_prompt(query, response)
    judge_output = judge_model_func(prompt)
    parsed_evaluation = parse_judge_response(judge_output)

    if "error" in parsed_evaluation:
        return {
            "dimensions": {},
            "overall_score": None,
            "evaluation_success": False,
            "error": parsed_evaluation["error"],
        }

    overall_score = parsed_evaluation.get("overall_score", 5)

    return {
        "dimensions": {
            "accuracy": parsed_evaluation.get("accuracy", {}),
            "helpfulness": parsed_evaluation.get("helpfulness", {}),
            "professionalism": parsed_evaluation.get("professionalism", {}),
            "clarity": parsed_evaluation.get("clarity", {}),
        },
        "overall_score": overall_score,
        "evaluation_success": True,
    }


def classify_response_quality(evaluation: Dict[str, Any]) -> str:
    """
    Classify response quality based on overall score.

    Categories:
    - "excellent": overall_score >= 9
    - "good": overall_score >= 7
    - "fair": overall_score >= 5
    - "poor": overall_score < 5

    Args:
        evaluation: Evaluation dictionary from evaluate_response_quality()

    Returns:
        Quality classification string

    Edge Cases:
        - If evaluation missing overall_score, return "unknown"
        - If overall_score is None or invalid, return "unknown"
    """
    if not evaluation:
        return "unknown"

    overall_score = evaluation.get("overall_score", None)

    if not overall_score or overall_score < 1 or overall_score > 10:
        return "unknown"

    if overall_score >= 9:
        return "excellent"
    elif overall_score >= 7:
        return "good"
    elif overall_score >= 5:
        return "fair"
    else:
        return "poor"


In [ ]:
"""
Module 3: Multi-Judge Consensus System

Implements robust evaluation framework using multiple independent judges with
consensus calculation, disagreement detection, and confidence scoring.
"""

from typing import List, Dict, Any, Callable
import numpy as np


def evaluate_with_multiple_judges(
    query: str,
    response: str,
    judge_functions: List[Callable[[str, str], Dict[str, Any]]],
) -> List[Dict[str, Any]]:
    """
    Collect independent evaluations from multiple judge functions.

    Each judge function should evaluate the same query-response pair independently
    without knowledge of other judges' scores. This prevents cascade failures where
    one incorrect judgment influences others.

    Args:
        query: Customer's original question
        response: AI-generated response to evaluate
        judge_functions: List of judge callables, each taking (query, response)
                        and returning evaluation dict with 'overall_score'

    Returns:
        List of evaluation dictionaries, one per judge

    Edge Cases:
        - If a judge function raises an error, catch it and mark that evaluation as failed
        - Continue with remaining judges even if one fails
        - Return partial results with error indication for failed judges
    """
    evaluations = []
    for judge in judge_functions:
        try:
            result = judge(query, response)
            evaluations.append({"result": result, "evaluation_failed": False})
        except Exception:
            evaluations.append({"result": result, "evaluation_failed": True})

    return evaluations


def calculate_consensus_score(
    evaluations: List[Dict[str, Any]], weights: List[float] = None
) -> float:
    """
    Calculate consensus score using weighted averaging of judge scores.

    Combines multiple judges' overall scores using weighted averaging. Default
    weights are equal for all judges, but custom weights can prioritize certain
    judges based on their historical accuracy.

    Formula:
    consensus = sum(score_i * weight_i) / sum(weight_i)

    Args:
        evaluations: List of evaluation dictionaries with 'overall_score'
        weights: Optional list of weights (same length as evaluations)
                Default: equal weights for all judges

    Returns:
        Weighted average consensus score (0-10 scale)

    Edge Cases:
        - Filter out failed evaluations (those without 'overall_score')
        - If all evaluations failed, return 0.0
        - If weights provided but wrong length, use equal weights
        - If weights don't sum to positive value, use equal weights
        - Weights should be normalized (sum to 1.0) before calculation
    """
    if not evaluations:
        return 0.0

    # Filter out failed evaluations
    valid_scores = [e["overall_score"] for e in evaluations if "overall_score" in e]
    if not valid_scores:
        return 0.0

    # If weights are not provided, use equal weights
    if weights is None or len(weights) != len(valid_scores):
        weights = [1.0] * len(valid_scores)

    # Normalize weights
    weight_sum = sum(weights)
    if weight_sum <= 0:
        weights = [1.0] * len(valid_scores)
        weight_sum = len(valid_scores)

    normalized_weights = [w / weight_sum for w in weights]

    # Calculate weighted average
    consensus = sum(
        score * weight for score, weight in zip(valid_scores, normalized_weights)
    )
    return consensus


def detect_disagreement(
    evaluations: List[Dict[str, Any]], threshold: float = 2.0
) -> bool:
    """
    Detect significant disagreement between judges.

    Calculates standard deviation of overall scores. High standard deviation
    indicates judges fundamentally disagree on quality, suggesting the case
    may require human review.

    Typical thresholds:
    - < 1.5: Strong agreement
    - 1.5-2.5: Moderate disagreement
    - > 2.5: Significant disagreement

    Args:
        evaluations: List of evaluation dictionaries with 'overall_score'
        threshold: Standard deviation threshold for flagging disagreement

    Returns:
        True if standard deviation >= threshold, False otherwise

    Edge Cases:
        - Filter out failed evaluations before calculating
        - If fewer than 2 valid evaluations, return False
        - If all scores are identical, std_dev = 0 (no disagreement)
    """
    if not evaluations:
        return False

    # Filter out failed evaluations
    valid_scores = [e["overall_score"] for e in evaluations if "overall_score" in e]
    if len(valid_scores) < 2:
        return False

    # Calculate standard deviation
    mean = sum(valid_scores) / len(valid_scores)
    squared_diffs = [(x - mean) ** 2 for x in valid_scores]
    std_dev = (sum(squared_diffs) / len(squared_diffs)) ** 0.5

    return std_dev >= threshold


def calculate_confidence(evaluations: List[Dict[str, Any]]) -> float:
    """
    Calculate evaluation confidence based on judge agreement.

    High agreement (low variance) indicates high confidence in the evaluation.
    Converts standard deviation to confidence score where lower variance means
    higher confidence.

    Formula:
    confidence = 1.0 / (1.0 + std_dev)

    This gives:
    - std_dev = 0 (perfect agreement) -> confidence = 1.0
    - std_dev = 1.0 -> confidence = 0.5
    - std_dev = 4.0 -> confidence = 0.2

    Args:
        evaluations: List of evaluation dictionaries with 'overall_score'

    Returns:
        Confidence score between 0 and 1 (1 = highest confidence)

    Edge Cases:
        - Filter out failed evaluations before calculating
        - If fewer than 2 valid evaluations, return 0.0
        - If all scores identical (std_dev = 0), return 1.0
    """
    if not evaluations:
        return 0.0

    # Filter out failed evaluations
    valid_scores = [e["overall_score"] for e in evaluations if "overall_score" in e]
    if len(valid_scores) < 2:
        return 0.0

    # Calculate standard deviation
    mean = sum(valid_scores) / len(valid_scores)
    squared_diffs = [(x - mean) ** 2 for x in valid_scores]
    std_dev = (sum(squared_diffs) / len(squared_diffs)) ** 0.5

    confidence = 1.0 / (1.0 + std_dev)

    return confidence
